In [14]:
import json

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

from shapely.geometry import Polygon

Load census variables and parse for relevant information

In [15]:
TARIFF_IMPACT_FILE = '../raw/tariffs/ada_tariff_counts_percents_9_29_2026.xlsx'

# Create mapping for characteristic IDs to column names
characteristic_mapping = {
    1: 'CHAR_POP21'  # Population, 2021
}

df_cen_cma_data = pd.read_csv('../../data/census/98-401-X2021012_English_CSV_data.csv', encoding='latin')
df_cen_cma_data = df_cen_cma_data[['DGUID', 'GEO_LEVEL', 'GEO_NAME', 'CHARACTERISTIC_ID', 'C1_COUNT_TOTAL']]

# Filter for only the characteristics we want
df_cen_cma_data = df_cen_cma_data[df_cen_cma_data['CHARACTERISTIC_ID'].isin(characteristic_mapping.keys())]

# Map characteristic IDs to column names
df_cen_cma_data['CHARACTERISTIC_COLUMN'] = df_cen_cma_data['CHARACTERISTIC_ID'].map(characteristic_mapping)

# Pivot to create separate columns for each characteristic
df_cen_cma_data = df_cen_cma_data.pivot_table(
    index=['DGUID', 'GEO_LEVEL', 'GEO_NAME'], 
    columns='CHARACTERISTIC_COLUMN', 
    values='C1_COUNT_TOTAL',
    aggfunc='first'
).reset_index()

# Flatten column names
df_cen_cma_data.columns.name = None

In [16]:
df_ada_cma_rel = pd.read_csv('../../data/census/ada_cma_relation.csv')
df_ada_cma_rel = df_ada_cma_rel.rename(columns={'CMADGUID_RMRIDUGD': 'CMADGUID', 'ADADGUID_ADAIDUGD': 'ADADGUID'})

In [17]:
# lcma000b21a_e is a folder containing the shapefile components of census metropolitan areas of Canada 2021
gdf_cma = gpd.read_file('../../data/census/lcma000b21a_e')
gdf_cma = gdf_cma[['CMAUID', 'DGUID', 'CMANAME', 'CMATYPE', 'PRUID', 'geometry']]

# Merge duplicates on DGUID: first row's attrs + unioned geometry
gdf_cma = (
    gdf_cma
    .groupby('DGUID', as_index=False)
    .agg({
        'CMAUID': 'first',
        'CMANAME': 'first',
        'CMATYPE': 'first',
        'PRUID': 'first',
        'geometry': lambda x: x.unary_union
    })
)

# Remove anything inside parentheses (and the parentheses themselves)
gdf_cma['CMANAME'] = gdf_cma['CMANAME'].str.replace(r"\s*\(.*?\)", "", regex=True).str.strip()

gdf_cma = gpd.GeoDataFrame(gdf_cma, geometry='geometry')
if gdf_cma.crs is None:
    gdf_cma.set_crs("EPSG:3347", inplace=True)


print(gdf_cma.CMATYPE.value_counts())

CMATYPE
D    102
B     41
K      9
Name: count, dtype: int64


Load tariff information and join to CMAs

In [21]:
# Load ADA-level tariff counts and percents
df_tariffs_ada_count = pd.read_excel(TARIFF_IMPACT_FILE, sheet_name='Counts').drop(columns=['geometry'])
df_tariffs_ada_pct = pd.read_excel(TARIFF_IMPACT_FILE, sheet_name='Percents')

In [23]:
df_tariffs_count_filtered = (
    df_tariffs_ada_count
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Group by CMADGUID and sum all tariff columns
tariff_columns_count = [col for col in df_tariffs_count_filtered.columns if col not in ['ADADGUID', 'CMADGUID']]
df_tariffs_cma_count = df_tariffs_count_filtered.groupby('CMADGUID')[tariff_columns_count].sum().reset_index()

print(f"Shape of CMA tariffs (counts) dataframe: {df_tariffs_cma_count.shape}")
df_tariffs_cma_count.head()

Shape of CMA tariffs (counts) dataframe: (152, 55)


,CMADGUID,Auto_B,Auto_E,Auto_C,Alum_B,Alum_E,Alum_C,Steel_B,Steel_E,Steel_C,...,CUSMA_C,Total_B,Total_E,Total_C,ScenBefore_B,ScenBefore_E,ScenBefore_C,ScenAfter_B,ScenAfter_E,ScenAfter_C
0,2021S0503001,7,41,147,17,105,228,14,95,218,...,2736,118,2046,2736,118,2030,2732,118,2046,2736
1,2021S0503205,35,708,1302,90,2087,2920,57,1222,1702,...,10846,421,7251,11833,414,6513,11106,421,7251,11833
2,2021S0503305,15,317,378,37,798,700,34,685,612,...,4440,191,4346,4977,186,3796,4607,191,4346,4977
3,2021S0503310,5,153,313,17,399,637,13,1204,1418,...,4071,142,3904,5200,142,3582,4752,142,3904,5200
4,2021S0503320,9,172,191,26,360,390,15,262,311,...,2414,132,1833,2567,132,1730,2393,132,1833,2567


In [24]:
# Melt percents and counts to long format, extract base and numeric suffix
def melt_tariffs(df, value_name, suffix_map=None):
    df_long = df.melt(id_vars=['ADADGUID'], var_name='tariff', value_name=value_name)
    df_long['base'] = df_long['tariff'].str.replace(r'_(1|2|3|B|E|C)$', '', regex=True)
    df_long['suffix'] = df_long['tariff'].str.extract(r'_([1-3BEC])$')[0]
    if suffix_map:
        df_long['suffix'] = df_long['suffix'].map(suffix_map)
    return df_long

suffix_map_counts = {'B': '1', 'E': '2', 'C': '3'}

df_pct_long = melt_tariffs(df_tariffs_ada_pct, 'percent')
df_count_long = melt_tariffs(df_tariffs_ada_count, 'count', suffix_map=suffix_map_counts)

# Merge percents and counts, add CMA IDs
df_merge = (
    df_pct_long
    .merge(df_count_long, on=['ADADGUID', 'base', 'suffix'], how='left')
    .merge(df_ada_cma_rel, on='ADADGUID', how='left')
    .dropna(subset=['CMADGUID'])
)

# Compute weighted percent and aggregate per CMA
df_cma = (
    df_merge.assign(weighted=lambda x: x['percent'] * x['count'])
    .groupby(['CMADGUID', 'base', 'suffix'], observed=True)
    .agg(total_weighted=('weighted', 'sum'), total_count=('count', 'sum'))
    .reset_index()
)
df_cma['cma_percent'] = (df_cma['total_weighted'] / df_cma['total_count']) * 100
df_cma.loc[df_cma['total_count'] == 0, 'cma_percent'] = np.nan

# Pivot to wide format
df_cma['colname'] = df_cma['base'] + '_' + df_cma['suffix']
df_tariffs_cma_pct = df_cma.pivot(index='CMADGUID', columns='colname', values='cma_percent').reset_index()

print(f"Shape of CMA tariffs (percents) dataframe: {df_tariffs_cma_pct.shape}")
df_tariffs_cma_pct.head()

Shape of CMA tariffs (percents) dataframe: (152, 55)


colname,CMADGUID,Alcohol_1,Alcohol_2,Alcohol_3,Alum_1,Alum_2,Alum_3,Auto_1,Auto_2,Auto_3,...,ScenAfter_3,ScenBefore_1,ScenBefore_2,ScenBefore_3,Steel_1,Steel_2,Steel_3,Total_1,Total_2,Total_3
0,2021S0503001,0.410149,0.249816,0.065670,0.569074,0.691774,0.229207,0.510324,0.208220,0.156783,...,2.631058,2.111118,7.512447,2.625433,0.744515,0.824444,0.220107,2.111118,7.708065,2.631058
1,2021S0503205,0.813391,0.378418,0.238185,1.273492,3.284860,1.252453,0.609318,1.689228,0.538974,...,5.024624,6.164088,6.181843,4.728353,0.966178,1.702845,0.713152,6.210002,6.692827,5.024624
2,2021S0503305,0.397276,0.694481,0.259956,1.162877,2.568543,0.916250,0.663729,0.847195,0.531749,...,6.745000,5.893414,8.706873,6.265466,1.525981,2.509847,0.831770,6.062326,9.286939,6.745000
3,2021S0503310,0.968335,5.982111,0.657352,1.065927,2.744047,1.133745,0.901604,3.344821,0.624873,...,8.883014,10.772117,9.166756,8.246035,1.251197,8.082384,2.241528,10.772117,9.334061,8.883014
4,2021S0503320,1.246377,0.683107,0.455458,1.389997,2.018435,0.822952,0.688483,2.629791,0.408295,...,5.994259,6.647012,16.268406,5.576130,1.651715,2.551230,0.701424,6.647012,15.992116,5.994259


In [25]:
# First, we need to aggregate census data from ADA to CMA level
df_cen_ada = df_cen_cma_data.rename(columns={'DGUID': 'ADADGUID'})

# Merge census ADA data with ADA-CMA relationship
df_cen_with_cma = df_cen_ada.merge(df_ada_cma_rel, on='ADADGUID', how='left')

# Aggregate population to CMA level
df_cen_cma = df_cen_with_cma.groupby('CMADGUID').agg({
    'GEO_NAME': 'first',  # replaced below with the proper CMA name
    'CHAR_POP21': 'sum'
}).reset_index()

# Merge with GDF to get proper names AND the CMA/CA type flag
gdf_cma_names = (gdf_cma[['DGUID', 'CMANAME', 'CMATYPE']]
                 .rename(columns={'DGUID': 'CMADGUID', 'CMANAME': 'GEO_NAME'}))
df_cen_cma = df_cen_cma.drop(columns=['GEO_NAME']).merge(gdf_cma_names, on='CMADGUID', how='left')

# Derive GEO_LEVEL from CMATYPE. Never hardcode this: 'B' is a census
# metropolitan area, 'D' and 'K' are census agglomerations.
CMA_CODE = 'B'
df_cen_cma['GEO_LEVEL'] = np.where(df_cen_cma.CMATYPE == CMA_CODE,
                                   'Census metropolitan area',
                                   'Census agglomeration')

assert (df_cen_cma.GEO_LEVEL == 'Census metropolitan area').sum() == 41, \
    df_cen_cma.GEO_LEVEL.value_counts()

# Now merge with tariff data
df_final_counts = df_cen_cma.merge(df_tariffs_cma_count, on='CMADGUID', how='inner')
df_final_percents = df_cen_cma.merge(df_tariffs_cma_pct, on='CMADGUID', how='inner')

df_final_counts = df_final_counts[df_final_counts['CMATYPE'] == 'B']
df_final_percents = df_final_percents[df_final_percents['CMATYPE'] == 'B']

print(f"Shape of final counts dataframe: {df_final_counts.shape}")
print(f"Shape of final percents dataframe: {df_final_percents.shape}")
print(df_final_counts.GEO_LEVEL.value_counts().to_string())

Shape of final counts dataframe: (41, 59)
Shape of final percents dataframe: (41, 59)
GEO_LEVEL
Census metropolitan area    41


In [26]:
# Prepare geometry data
gdf_cma_geom = gpd.GeoDataFrame(
    gdf_cma[['DGUID', 'geometry']].rename(columns={'DGUID': 'CMADGUID'}),
    geometry='geometry',
    crs=gdf_cma.crs
)

# ---- COUNTS ----
gdf_final_counts_full = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

# Centroids for counts
gdf_cma_centroids = gdf_cma_geom.copy()
gdf_cma_centroids['geometry'] = gdf_cma_centroids.geometry.centroid
gdf_cma_centroids = gdf_cma_centroids.to_crs('EPSG:4326')

gdf_final_counts_centroids = gpd.GeoDataFrame(
    df_final_counts.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save counts
gdf_final_counts_full.to_file('../../data/cma/cma_tariffs_counts_full_geometry.gpkg', driver='GPKG')

# Save counts centroids as CSV: convert geometry to WKT only for the CSV copy
df_counts_centroids_csv = gdf_final_counts_centroids.copy()
df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_counts_centroids_csv.to_csv('../../data/cma/cma_tariffs_counts_centroids.csv', index=False)

# ---- PERCENTS ----
gdf_final_percents_full = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_geom, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs=gdf_cma.crs
)

gdf_final_percents_centroids = gpd.GeoDataFrame(
    df_final_percents.merge(gdf_cma_centroids, on='CMADGUID', how='inner'),
    geometry='geometry',
    crs='EPSG:4326'
)

# Save percents
gdf_final_percents_full.to_file('../../data/cma/cma_tariffs_percents_full_geometry.gpkg', driver='GPKG')

# Save percents centroids as CSV: convert geometry to WKT only for the CSV copy
df_percents_centroids_csv = gdf_final_percents_centroids.copy()
df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)
df_percents_centroids_csv.to_csv('../../data/cma/cma_tariffs_percents_centroids.csv', index=False)

# Print shapes
print(f"Counts full geometry shape: {gdf_final_counts_full.shape}")
print(f"Percents full geometry shape: {gdf_final_percents_full.shape}")
print(f"Counts centroids shape: {gdf_final_counts_centroids.shape}")
print(f"Percents centroids shape: {gdf_final_percents_centroids.shape}")


C:\Users\yihoi\AppData\Local\Temp\ipykernel_18784\3454657186.py:31: UserWarning: Geometry column does not contain geometry.
  df_counts_centroids_csv['geometry'] = df_counts_centroids_csv.geometry.apply(lambda geom: geom.wkt)


Counts full geometry shape: (41, 60)
Percents full geometry shape: (41, 60)
Counts centroids shape: (41, 60)
Percents centroids shape: (41, 60)


C:\Users\yihoi\AppData\Local\Temp\ipykernel_18784\3454657186.py:52: UserWarning: Geometry column does not contain geometry.
  df_percents_centroids_csv['geometry'] = df_percents_centroids_csv.geometry.apply(lambda geom: geom.wkt)


In [27]:
counts_path = '../../data/cma/cma_tariffs_counts_centroids.csv'
percents_path = '../../data/cma/cma_tariffs_percents_centroids.csv'

counts_json_path = '../../data/cma/cma_tariffs_counts_centroids.json'
percents_json_path = '../../data/cma/cma_tariffs_percents_centroids.json'

def csv_to_json_records(csv_path):
    df = pd.read_csv(csv_path)

    # Replace all NaN, NaT, and inf values with None so they become valid JSON null
    df = df.replace({np.nan: None, np.inf: None, -np.inf: None})

    return df.to_dict(orient='records')

counts_records = csv_to_json_records(counts_path)
percents_records = csv_to_json_records(percents_path)

with open(counts_json_path, 'w', encoding='utf-8') as f:
    json.dump(counts_records, f, ensure_ascii=False, indent=4)
with open(percents_json_path, 'w', encoding='utf-8') as f:
    json.dump(percents_records, f, ensure_ascii=False, indent=4)

print(f'Saved counts JSON: {counts_json_path} (rows={len(counts_records)})')
print(f'Saved percents JSON: {percents_json_path} (rows={len(percents_records)})')


Saved counts JSON: ../../data/cma/cma_tariffs_counts_centroids.json (rows=41)
Saved percents JSON: ../../data/cma/cma_tariffs_percents_centroids.json (rows=41)


In [28]:
df_cma_pcts = pd.read_csv(percents_path)
df_cma_pcts = df_cma_pcts[df_cma_pcts['GEO_LEVEL'] == 'Census metropolitan area']
df_cma_pcts.describe(percentiles=[0.2, 0.4, 0.6, 0.8]).round(2).to_csv('../../data/cma/cma_tariffs_percents_variance.csv')

In [31]:
# ============================================================
# EXPORT: counts + percents to one workbook, with readable
# indicator names instead of the _1/_2/_3 and _B/_E/_C suffixes.
# ============================================================
import re

pd.io.formats.excel.ExcelFormatter.header_style = None

OUTPUT_XLSX = '../../data/cma/cma_tariffs_counts_and_percents.xlsx'

# Counts and percents use different suffix letters for the same three
# indicators, so both map onto one shared vocabulary.
SUFFIX_MAP = {
    '1': 'Business', 'B': 'Business',
    '2': 'EmployeeWork', 'E': 'EmployeeWork',
    '3': 'EmployeeHome', 'C': 'EmployeeHome',
}

def rename_suffixes(df):
    """Rename `<base>_<suffix>` -> `<base>_<IndicatorName>`.

    Splits on the FINAL underscore only, so bases that themselves contain
    underscores (S338_Tar, S338_Exc, S338_Tot) survive intact. Columns with
    no recognised suffix (CMADGUID, GEO_NAME, CHAR_POP21, geometry, ...) are
    left untouched.
    """
    renames = {}
    for col in df.columns:
        m = re.match(r'^(.*)_([123BEC])$', col)
        if m:
            base, suffix = m.group(1), m.group(2)
            renames[col] = f'{base}_{SUFFIX_MAP[suffix]}'
    return df.rename(columns=renames)

counts_out = rename_suffixes(df_final_counts).drop(columns=['geometry'], errors='ignore')
percents_out = rename_suffixes(df_final_percents).drop(columns=['geometry'], errors='ignore')

# Fail loudly if an indicator never made it through, rather than shipping a
# silently two-thirds-complete workbook (the _3 / _C problem).
for label, df in [('Counts', counts_out), ('Percents', percents_out)]:
    found = {m.group(1) for c in df.columns
             if (m := re.search(r'_(Business|EmployeeWork|EmployeeHome)$', c))}
    missing = {'Business', 'EmployeeWork', 'EmployeeHome'} - found
    if missing:
        print(f'WARNING: {label} is missing indicator(s): {sorted(missing)}')
    print(f'{label}: {df.shape[0]} rows, {df.shape[1]} cols, indicators: {sorted(found)}')

with pd.ExcelWriter(OUTPUT_XLSX, engine='openpyxl') as writer:
    counts_out.to_excel(writer, sheet_name='Counts', index=False)
    percents_out.to_excel(writer, sheet_name='Percents', index=False)

print(f'\nSaved {OUTPUT_XLSX}')

Counts: 41 rows, 59 cols, indicators: ['Business', 'EmployeeHome', 'EmployeeWork']
Percents: 41 rows, 59 cols, indicators: ['Business', 'EmployeeHome', 'EmployeeWork']

Saved ../../data/cma/cma_tariffs_counts_and_percents.xlsx


In [29]:
# old = pd.read_excel('../raw/tariff-impacts-ada-data.xlsx', sheet_name='Percents')
# new = pd.read_excel('../raw/tariff-impacts-ada-data_9_7_2026.xlsx', sheet_name='Percents')
# m = old.merge(new, on='ADADGUID', suffixes=('_old','_new'))
# print((m['CUSMA_3_new'] - m['CUSMA_3_old']).describe())
# print((m['Total_3_new'] - m['Total_3_old']).describe())